In [ ]:
import sys

sys.path.append("..")
from src.model.geoclip import GeoCLIP
from src.model.g3 import G3
from src.utils import (
    build_index,
    add_record_to_index,
    save_index,
    get_device,
    read_index,
)
import polars as pl
import torch
from torch.nn import functional as F
from tqdm import tqdm
from src.model.backbones import load_backbone
import json

In [2]:
DEVICE = get_device()

In [31]:
index, meta = read_index("../index/twostep-geoclip-hnsw")
gps_index, _ = read_index("../index/twostep-geoclip-hnsw", prefix="gps")
meta = meta["metadata"]

In [7]:
backbone = load_backbone("geoclip", DEVICE)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
with open(
    "/mnt/yokoyamalab-nas/gldv2-full/csv/queries/test_templated_queries.json", "r"
) as f:
    data = json.loads(f.read())["data"]
    data = [d for d in data if len(d["relevant_ids"]) >= 5]

In [44]:
dd = [d for d in data if d["country"] == "Japan"]
query = dd[0]["text"]
query_embed = backbone.encode_text([query], 1).numpy()

encode text:   0%|          | 0/1 [00:00<?, ?it/s]

In [46]:
dd[0]

{'text': 'a amusement park located in Japan',
 'category': 'amusement park',
 'country': 'Japan',
 'region': 'Asia',
 'relevant_ids': ['1efb7a7a76f1b239',
  '4a73d6cbd37ba12e',
  '54ce9b101f7c37cb',
  '5d52f854e7fae035',
  '8b508215bcf6ed1e',
  'a7a65e469c25604a',
  'ac0caaef99dedd6e',
  'f169231b9a80d548']}

In [47]:
_, indices = gps_index.search(query_embed, 10_000)
indices = indices.reshape(-1).tolist()
first_pass = [meta[i] for i in indices]

In [ ]:
import faiss

sel = faiss.IDSelectorArray(indices)
param = faiss.SearchParameters(sel=sel)
_, indices2 = index.search(query_embed, 100, params=param)

In [84]:
second_pass = [meta[i] for i in indices2[indices2 != -1] if i != -1]
second_pass

[{'id': 'e4ab5918e1560eae',
  'landmark_id': 9998,
  'wikimedia_url': 'https://commons.wikimedia.org/wiki/Category:Denpark',
  'src': 'train',
  'geohack_url': 'https://geohack.toolforge.org/geohack.php?pagename=Category:Denpark&params=34.930217_N_137.060744_E_globe:Earth_&language=en',
  'latitude': 34.930217,
  'longitude': 137.060744,
  'poi_name': 'Denpark',
  'wikidata_id': 'Q11450483',
  'instance_tag': 'park,botanical garden',
  'category': 'park',
  'cluster_id': '2,27',
  'scene_pred': 'exterior',
  'country_code': 'JP',
  'country': 'Japan',
  'region': 'Asia',
  'subregion': 'Eastern Asia',
  'city': 'Anjo',
  'split': 'test',
  'caption': 'A colorful miniature train rides along a grassy path in a park, a leisure spot in Japan.'},
 {'id': '167c2934336bc406',
  'landmark_id': 164861,
  'wikimedia_url': 'https://commons.wikimedia.org/wiki/Category:Mosaic_Garden',
  'src': 'train',
  'geohack_url': 'https://geohack.toolforge.org/geohack.php?pagename=Category:Mosaic_Garden&param

In [86]:
gt = dd[0]["relevant_ids"]
pred = [d["id"] for d in second_pass]

In [89]:
normalizer = min(len(gt), 100)
h, cs = 0, 0

for i, img_id in enumerate(pred[:100], start=1):
    if img_id in gt:
        h += 1
        cs += h/i

cs / normalizer

0.0

In [90]:
sum(1 for img_id in pred[:100] if img_id in gt)

0